# GitHub Issues Agent

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSV-AI/agent-playground/blob/dev/notebooks/github_issues.ipynb)

## Description

This notebook replicates the functionality of the `src/agent_playground/github_issues` agent, demonstrating GitHub repository issue analysis using PydanticAI and GitHub MCP tools.


## Setup

Before running this notebook, ensure you have the required dependencies installed and your OpenRouter API key configured.

### Install required Python dependencies

In [ ]:
!pip install pydantic-ai pydantic python-dotenv

### Import necessary libraries

In [ ]:
import os
import asyncio
from pydantic import BaseModel, Field
from pydantic_ai import Agent, AgentRunResult
from pydantic_ai.mcp import MCPServerStreamableHTTP
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openrouter import OpenRouterProvider
import logfire
from typing import List, Any
from datetime import datetime
from pathlib import Path
from google.colab import userdata

## Define Data Models

These models are identical to those found in `src/agent_playground/github_issues/models.py`.

In [ ]:
class GitHubIssue(BaseModel):
    """Represents a single GitHub issue."""
    title: str = Field(description="The title of the GitHub issue.")
    url: str = Field(description="The URL of the GitHub issue.")
    state: str = Field(description="The current state of the issue (e.g., 'open', 'closed').")
    updated_at: datetime = Field(description="The last time the issue was updated.")
    labels: List[str] = Field(description="A list of labels associated with the issue.")
    author: str = Field(description="The GitHub username of the issue author.")

class GitHubIssueReport(BaseModel):
    """A report of GitHub issues."""
    issues: List[GitHubIssue] = Field(description="A list of GitHub issues found.")
    repository: str = Field(description="The name of the repository the issues were fetched from.")
    owner: str = Field(description="The owner of the repository.")
    report_summary: str = Field(description="A summary of the issues found and reported.")
    error_message: str = Field(default="", description="An error message if the report generation failed, otherwise empty.")

## Configure LLM and MCP Server

Configure PydanticAI to use OpenRouter with an OpenAI-compatible model and set up the GitHub MCP server.

In [ ]:
# --- Configuration ---
# Since OpenRouter has a unified API, we can use the OpenAIChatModel with a custom provider.
OPENROUTER_MODEL = "openai/gpt-4o-mini"
OPENROUTER_KEY = userdata.get('OPENROUTER_API_KEY') # For Colab secrets

# 1. Configure the LLM for OpenRouter
_model = OpenAIChatModel(
    OPENROUTER_MODEL,
    provider=OpenRouterProvider(
        api_key=OPENROUTER_KEY
    ),
)

# 2. Configure the GitHub MCP Server
# Using this approach instead of load_mcp_servers to directly define the MCP server
# because we need to pass environment variables through the configuration.
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') # For Colab secrets

_toolsets = [
    MCPServerStreamableHTTP("https://api.githubcopilot.com/mcp",
        headers={"Authorization": f"Bearer {GITHUB_TOKEN}"})
]

## System Prompt

The system prompt guides the agent's behavior. This is directly from `src/agent_playground/github_issues/system_prompt.md`.

In [ ]:
system_prompt_text = """
# Expert GitHub Issues Agent

You are a specialized GitHub Issues Agent. Your primary objective is to autonomously fulfill the user's request using the provided GitHub toolset.

# Core Directives & Procedure

1. Strict Tool Use: You MUST use the GitHub tools for all repository interactions, analysis, and data extraction tasks related to issues. Do not attempt to guess or hallucinate content.
2. Initial Action: Begin by understanding the user's request, which will typically involve querying GitHub issues related to the current project or repository.
3. Dynamic Interaction: If required, use appropriate GitHub tools to gather more context (e.g., list issues, filter by status, get issue details) before formulating a response.
4. Information Extraction: Use the most precise GitHub tool available (e.g., `github_list_issues`, `github_get_issue`) to gather the requested data.

# Exit Strategy and Error Handling

Your exit strategy must be based on the outcome of your operations:

1. Success: If you successfully gather all the requested data, format it clearly and provide it as the response.
2. Failure/Error: If you encounter any of the following issues, you MUST ABORT the task and return a clear error message:
  - A GitHub tool reports an error (e.g., repository not found, invalid API token, rate limit exceeded).
  - You are unable to extract relevant data despite successful tool calls.
  - You exhaust the maximum number of tool call retries.
3. Failure Response: Upon failure, clearly explain the problem encountered.

# Final Output Requirement

Your final action MUST be to provide a comprehensive and clear response to the user's query, always prioritizing accuracy and relevance based on the GitHub repository's issues. Do not include any conversational wrapper text in the final output beyond what is necessary to present the information.
"""

## Initialize the Agent

Create the Agent instance with the configured LLM, output type, toolsets, and system prompt.

In [ ]:
agent = Agent(
    model=_model,
    output_type=GitHubIssueReport,
    toolsets=_toolsets,
    system_prompt=system_prompt_text,
)

## User Prompt

This is the task the agent will perform, pulled directly from `src/agent_playground/github_issues/user_prompt.md`.

In [ ]:
user_prompt_text = """
Please find all issues related to the current project (HSV-AI/agent-playground).
Report on the 5 most recently updated issues, providing their title, URL, Issue Description,
and current status.
"""

## Run the Agent

Execute the agent with the user prompt.

In [ ]:
# configure logfire (optional)
LOGFIRE_TOKEN = os.environ.get('LOGFIRE_TOKEN')
if LOGFIRE_TOKEN:
  logfire.configure(token=LOGFIRE_TOKEN)
  logfire.instrument_pydantic_ai()

print(f"Running Agent Task: {user_prompt_text}\n")

# Run the agent, specifying the desired output structure
result = await agent.run(
    user_prompt_text,
)

print("Task Complete! GitHub Issue Report:")
print(f"Repository: {result.output.owner}/{result.output.repository}")
print(f"Summary: {result.output.report_summary}")
for i, issue in enumerate(result.output.issues):
    print(f"Issue {i+1}:")
    print(f"  Title: {issue.title}")
    print(f"  URL: {issue.url}")
    print(f"  State: {issue.state}")
    print(f"  Updated At: {issue.updated_at}")
    print(f"  Labels: {', '.join(issue.labels)}")
    print(f"  Author: {issue.author}")
if result.output.error_message:
    print(f"Error: {result.output.error_message}")